# 🫁 Pneumonia Detection — ALL-IN-ONE
### EfficientNetB0 Feature Extraction + Linear SVM
Runs end-to-end: download → extract → train → evaluate → save model.

In [ ]:
# ── STEP 1: Kaggle Credentials & Dataset Download ──────────
import os

os.environ['KAGGLE_USERNAME'] = "YOUR_KAGGLE_USERNAME"   # ← Replace
os.environ['KAGGLE_KEY']      = "YOUR_KAGGLE_KEY"        # ← Replace

!pip install -q kaggle fpdf joblib opencv-python-headless

print('📥 Downloading Chest X-Ray dataset from Kaggle...')
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia

print('📦 Unzipping dataset...')
!unzip -q chest-xray-pneumonia.zip
print("✅ Dataset ready! 'chest_xray/' folder is now available.")

In [ ]:
# ── STEP 2: Import Libraries ───────────────────────────────
import numpy as np
import joblib
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.preprocessing.image import img_to_array, load_img
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

print(f'TensorFlow version: {tf.__version__}')
print('✅ All libraries imported successfully.')

In [ ]:
# ── STEP 3: Helper Functions ───────────────────────────────
def load_and_preprocess_image(image_path, target_size=(224, 224)):
    img = load_img(image_path, target_size=target_size)
    img_array = img_to_array(img) / 255.0
    return np.expand_dims(img_array, axis=0)

def get_data_labels(directory):
    filepaths, labels = [], []
    for label_name in ['NORMAL', 'PNEUMONIA']:
        label_val = 0 if label_name == 'NORMAL' else 1
        class_dir = os.path.join(directory, label_name)
        if not os.path.exists(class_dir):
            print(f'⚠️  Folder not found: {class_dir}')
            continue
        for img_name in os.listdir(class_dir):
            if img_name.lower().endswith(('.jpeg', '.jpg', '.png')):
                filepaths.append(os.path.join(class_dir, img_name))
                labels.append(label_val)
    return filepaths, np.array(labels)

print('✅ Helper functions defined.')

In [ ]:
# ── STEP 4: Load Dataset Paths ─────────────────────────────
TRAIN_DIR = 'chest_xray/train'
TEST_DIR  = 'chest_xray/test'

print('🗂️  Indexing images...')
train_files, train_labels = get_data_labels(TRAIN_DIR)
test_files,  test_labels  = get_data_labels(TEST_DIR)

print(f'Training images : {len(train_files)}')
print(f'  Normal        : {int(np.sum(train_labels == 0))}')
print(f'  Pneumonia     : {int(np.sum(train_labels == 1))}')
print(f'Testing  images : {len(test_files)}')
print(f'  Normal        : {int(np.sum(test_labels == 0))}')
print(f'  Pneumonia     : {int(np.sum(test_labels == 1))}')

In [ ]:
# ── STEP 5: Feature Extraction using EfficientNetB0 ────────
# Note: This step takes ~5-10 minutes. Enable GPU in Runtime > Change runtime type.
print('🧠 Loading EfficientNetB0 (ImageNet weights, no top layer)...')
feature_extractor = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    pooling='avg',
    input_shape=(224, 224, 3)
)
print(f'Output feature vector size: {feature_extractor.output_shape[1]}')

def extract_features_batch(files, split_name=''):
    features = []
    total = len(files)
    for i, f in enumerate(files):
        img  = load_and_preprocess_image(f)
        feat = feature_extractor.predict(img, verbose=0)
        features.append(feat.flatten())
        if i % 200 == 0:
            print(f'  [{split_name}] {i}/{total} processed...')
    print(f'  [{split_name}] {total}/{total} done ✅')
    return np.array(features)

print('\n⚙️  Extracting features — Training set...')
X_train = extract_features_batch(train_files, 'Train')

print('\n⚙️  Extracting features — Testing set...')
X_test  = extract_features_batch(test_files, 'Test')

print(f'\nX_train shape: {X_train.shape}')
print(f'X_test  shape: {X_test.shape}')

In [ ]:
# ── STEP 6: Train Linear SVM ───────────────────────────────
print('🏋️  Training Linear SVM classifier...')
svm = SVC(kernel='linear', probability=True, random_state=42)
svm.fit(X_train, train_labels)
print('✅ Training complete!')

In [ ]:
# ── STEP 7: Evaluate Results ───────────────────────────────
y_pred = svm.predict(X_test)

acc = accuracy_score(test_labels, y_pred) * 100
print('=' * 40)
print(f'  ✅ Accuracy : {acc:.2f}%')
print('=' * 40)
print('\n📋 Classification Report:')
print(classification_report(test_labels, y_pred, target_names=['NORMAL', 'PNEUMONIA']))

In [ ]:
# ── STEP 8: Confusion Matrix ───────────────────────────────
cm = confusion_matrix(test_labels, y_pred)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'Pneumonia'],
            yticklabels=['Normal', 'Pneumonia'])
plt.title('Confusion Matrix — EfficientNetB0 + Linear SVM', fontsize=13)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()
print("📊 Saved as 'confusion_matrix.png'")

In [ ]:
# ── STEP 9: Save Model ─────────────────────────────────────
MODEL_PATH = 'pneumonia_svm_model.pkl'
joblib.dump(svm, MODEL_PATH)
print(f"💾 Model saved as '{MODEL_PATH}'")
print('👉 Download from Colab sidebar → Files panel → right-click → Download')
print('   Place in your Streamlit project folder to run the dashboard.')